# N00: Path Exploration

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxFields2026-TrainingData-part-05/release_20260414/Training_retrospective/T2W/5T/R_T2W_5T_0506.nii
/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxFields2026-TrainingData-part-05/release_20260414/Training_retrospective/T2W/5T/R_T2W_5T_0511.nii
/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxFields2026-TrainingData-part-05/release_20260414/Training_retrospective/T2W/5T/R_T2W_5T_0505.nii
/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxFields2026-TrainingData-part-05/release_20260414/Training_retrospective/T2W/7T/R_T2W_7T_0757.nii
/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxFields2026-TrainingData-part-05/release_20260414/Training_retrospective/T2W/7T/R_T2W_7T_0770.nii
/kaggle/input/datasets/mayaracaa/mrixfields3/MRIxFields2026-TrainingData-part-05/MRIxField

## Save to a Text File (.txt)

In [3]:
import os

output_file = '/kaggle/working/mri_files_list.txt'

with open(output_file, 'w') as f:
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            # Write each file path to a new line
            f.write(os.path.join(dirname, filename) + '\n')

print(f"Successfully saved all file paths to {output_file}")

Successfully saved all file paths to /kaggle/working/mri_files_list.txt


## Save to a Pandas DataFrame/CSV (Best for Machine Learning)

In [4]:
import os
import pandas as pd

file_paths = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.nii') or filename.endswith('.nii.gz'):
            file_paths.append(os.path.join(dirname, filename))

# Create a DataFrame and save to CSV
df = pd.DataFrame({'file_path': file_paths})
df.to_csv('/kaggle/working/mri_file_paths.csv', index=False)

print(f"Successfully saved {len(file_paths)} file paths to /kaggle/working/mri_file_paths.csv")

Successfully saved 2109 file paths to /kaggle/working/mri_file_paths.csv


## Pair Data

In [5]:
import pandas as pd
import numpy as np
import re
import os
def parse_filename(path):
    """Extract metadata from filename: [Cohort]_[Modality]_[FieldStrength]_[SubjectID].nii"""
    basename = os.path.basename(path).replace('.nii.gz', '').replace('.nii', '')
    parts = basename.split('_')
    if len(parts) >= 4:
        return parts[0], parts[1], parts[2], parts[3]
    return None, None, None, None
def main():
    print("Loading mri_file_paths.csv...")
    df = pd.read_csv('/kaggle/working/mri_file_paths.csv')
    
    print("Parsing filenames...")
    df['cohort'], df['modality'], df['field_strength'], df['subject_id'] = zip(*df['file_path'].map(parse_filename))
    
    # Drop rows where parsing failed
    df = df.dropna(subset=['subject_id'])
    
    # Check for duplicates per (subject, modality, field_strength)
    duplicates = df.duplicated(subset=['subject_id', 'modality', 'field_strength'])
    if duplicates.any():
        print(f"Warning: Found {duplicates.sum()} duplicate entries. Keeping the first occurrence.")
        df = df.drop_duplicates(subset=['subject_id', 'modality', 'field_strength'])

    print("Pivoting data to create wide matrix...")
    # Pivot so each row is a unique (subject_id, modality, cohort) and columns are field strengths
    wide_df = df.pivot(
        index=['subject_id', 'modality', 'cohort'],
        columns='field_strength',
        values='file_path'
    ).reset_index()
    
    # Save the master wide dataframe
    master_out = '/kaggle/working/paired_mri_data_master.csv'
    wide_df.to_csv(master_out, index=False)
    print(f"Saved master paired data to {master_out} (Rows: {len(wide_df)})")
    
    # Generate common translation pairs
    field_strengths = df['field_strength'].unique()
    pairs_to_extract = [
        ('1.5T', '3T'),
        ('3T', '7T'),
        ('1.5T', '7T'),
        ('0.1T', '1.5T'),
        ('5T', '7T')
    ]
    
    # Only keep pairs where both columns actually exist in the dataframe
    existing_columns = wide_df.columns
    for src, tgt in pairs_to_extract:
        if src in existing_columns and tgt in existing_columns:
            # Filter rows where BOTH source and target paths are not null
            pair_df = wide_df.dropna(subset=[src, tgt]).copy()
            
            # Keep only relevant columns
            cols_to_keep = ['subject_id', 'modality', 'cohort', src, tgt]
            pair_df = pair_df[cols_to_keep]
            
            # Rename for generic src/tgt loading if needed
            pair_df = pair_df.rename(columns={src: 'source_path', tgt: 'target_path'})
            pair_df['source_domain'] = src
            pair_df['target_domain'] = tgt
            
            out_file = f"/kaggle/working/pair_{src}_to_{tgt}.csv"
            pair_df.to_csv(out_file, index=False)
            print(f"Saved {out_file} (Rows: {len(pair_df)})")
if __name__ == '__main__':
    main()

Loading mri_file_paths.csv...
Parsing filenames...
Pivoting data to create wide matrix...
Saved master paired data to /kaggle/working/paired_mri_data_master.csv (Rows: 1980)
Saved pair_1.5T_to_3T.csv (Rows: 9)
Saved pair_3T_to_7T.csv (Rows: 9)
Saved pair_1.5T_to_7T.csv (Rows: 9)
Saved pair_0.1T_to_1.5T.csv (Rows: 0)
Saved pair_5T_to_7T.csv (Rows: 9)
